# 04 - Historical Elasticity Estimation

**Goal:** Estimate how sensitive lock behavior is to portfolio pricing/rate competitiveness.

This notebook moves the project from assumption-based scenario analysis into a model-estimated elasticity framework.

Business question:

> If we reduce portfolio rate margin by **0.125%**, how much should lock conversion improve, and is the lift large enough to offset lower margin?

For now, this uses the synthetic `mortgage_data.parquet` created in Notebook 01. Later, the same framework can be applied to real historical lock data.


## 1. Imports and Display Settings

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss, classification_report

pd.options.display.float_format = "{:,.4f}".format
RANDOM_STATE = 42


## 2. Load Historical / Synthetic Data

In a real production version, this section would be replaced with actual portfolio pricing and lock history.

Minimum required fields:

- `rate_diff`: offered rate minus market/benchmark rate  
- `locked`: 1 if borrower locked, 0 otherwise  
- `loan_amount`
- borrower / loan controls such as FICO, LTV, purpose, occupancy, market condition


In [2]:
df = pd.read_parquet("mortgage_data.parquet", engine="pyarrow")

print(df.shape)
df.head()


(1500000, 14)


,fico,ltv,loan_amount,purpose,occupancy,market_volatility,market_rate,offered_rate,rate_diff,rate_cut_25bps,lock_prob,locked,fico_bucket,ltv_bucket
0,722,97.0000,"491,983.0000",refinance,primary,high,6.2500,6.2500,0.0000,0,0.3034,0,700-739,90+
1,799,86.0000,"451,628.0000",refinance,primary,low,6.3750,6.5000,0.1250,0,0.3556,0,780+,80-90
2,712,78.6000,"753,610.0000",refinance,investment,low,6.1250,6.1250,0.0000,0,0.3216,0,700-739,70-80
3,634,54.3000,"345,063.0000",cash_out,investment,low,5.7500,5.7500,0.0000,0,0.3641,1,600-659,<60
4,726,66.8000,"756,271.0000",purchase,investment,low,7.3750,7.2500,-0.1250,0,0.6387,1,700-739,60-70


## 3. Basic Sanity Checks

In [3]:
required_cols = [
    "fico", "ltv", "loan_amount", "purpose", "occupancy", "market_volatility",
    "rate_diff", "locked"
]

missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("Overall lock rate:", round(df["locked"].mean(), 4))
print("\nLock rate by rate_diff:")
display(df.groupby("rate_diff")["locked"].agg(["count", "mean"]).sort_index())


Overall lock rate: 0.5091

Lock rate by rate_diff:


,count,mean
rate_diff,,
-0.3750,72914,0.8947
-0.2500,147701,0.8074
-0.1250,297785,0.6687
0.0000,608068,0.4704
0.1250,224273,0.3039
0.2500,149259,0.1735


## 4. Define Model Features

The key variable is `rate_diff`.

Interpretation:

```text
rate_diff = offered_rate - market_rate
```

So:

```text
negative rate_diff = better than market for borrower
positive rate_diff = worse than market for borrower
```

A margin reduction of 0.125 should be represented as:

```python
scenario_rate_diff = rate_diff - 0.125
```


In [4]:
target = "locked"

numeric_features = [
    "rate_diff",
    "fico",
    "ltv",
    "loan_amount",
]

categorical_features = [
    "purpose",
    "occupancy",
    "market_volatility",
]

model_cols = numeric_features + categorical_features

X = df[model_cols].copy()
y = df[target].astype(int)

X.head()


,rate_diff,fico,ltv,loan_amount,purpose,occupancy,market_volatility
0,0.0000,722,97.0000,"491,983.0000",refinance,primary,high
1,0.1250,799,86.0000,"451,628.0000",refinance,primary,low
2,0.0000,712,78.6000,"753,610.0000",refinance,investment,low
3,0.0000,634,54.3000,"345,063.0000",cash_out,investment,low
4,-0.1250,726,66.8000,"756,271.0000",purchase,investment,low


## 5. Optional Sampling for Speed

The synthetic dataset may be large. Sampling keeps notebook development fast.

For final analysis, set `MAX_ROWS = None`.


In [5]:
MAX_ROWS = 300_000

if MAX_ROWS is not None and len(X) > MAX_ROWS:
    sample_idx = X.sample(MAX_ROWS, random_state=RANDOM_STATE).index
    X_model = X.loc[sample_idx].copy()
    y_model = y.loc[sample_idx].copy()
else:
    X_model = X.copy()
    y_model = y.copy()

print("Modeling rows:", len(X_model))
print("Modeling lock rate:", round(y_model.mean(), 4))


Modeling rows: 300000
Modeling lock rate: 0.5083


## 6. Train / Test Split

This is not an A/B test yet. This is a historical response model.

The goal is to estimate the relationship between pricing competitiveness and probability of lock while controlling for other borrower/loan characteristics.


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_model
)

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train lock rate:", round(y_train.mean(), 4))
print("Test lock rate:", round(y_test.mean(), 4))


Train rows: 225000
Test rows: 75000
Train lock rate: 0.5083
Test lock rate: 0.5083


## 7. Fit Logistic Regression Model

Why logistic regression?

Because the outcome is binary:

```text
locked = 1 or 0
```

The model estimates:

```text
P(lock = 1 | pricing, FICO, LTV, purpose, occupancy, market condition)
```

This gives us a statistically grounded way to estimate pricing sensitivity.


In [7]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ],
    remainder="drop"
)

logit_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", logit_model)
    ]
)

model.fit(X_train, y_train)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['rate_diff', 'fico', 'ltv',
                                                   'loan_amount']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['purpose', 'occupancy',
                                                   'market_volatility'])])),
                ('model', LogisticRegression(max_iter=1000))])

## 8. Model Evaluation

This is not about perfect prediction yet.  

For this exercise, the main purpose is estimating directional pricing sensitivity.


In [8]:
test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.50).astype(int)

auc = roc_auc_score(y_test, test_prob)
accuracy = accuracy_score(y_test, test_pred)
brier = brier_score_loss(y_test, test_prob)

print("AUC:", round(auc, 4))
print("Accuracy:", round(accuracy, 4))
print("Brier score:", round(brier, 4))

print("\nClassification Report:")
print(classification_report(y_test, test_pred))


AUC: 0.7485
Accuracy: 0.6803
Brier score: 0.2033

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.67      0.67     36878
           1       0.69      0.69      0.69     38122

    accuracy                           0.68     75000
   macro avg       0.68      0.68      0.68     75000
weighted avg       0.68      0.68      0.68     75000



In [ ]:
# AUC Area under the ROC Curve
# how well the model separates loans likely to lock vs not lock
# 0.5 random guessing, 0.70 - 0.80 decent good, 0.80+ strong, 1 perfect

# Accuracy overall percentage of correct predictions (both locked and not locked)
# (TP + TN) / (TP + TN + FP + FN)

# * TP = correctly predicted locks
# * TN = correctly predicted non-locks
# * FP = false positives
# * FN = false negatives

# Brier Score, how accurate the predictions probabilities are
# 0 = perfect, 0.25 weak random ish for balance binary

# Precision: of all loans predicted to lock, how many actually locked
# TP / (TP + FP)

# Recall: of all actual locks, the model sufficefully idenitified how many
# TP / (TP + FN)

# F1 Score: combination of precision and recall, harmonic mean

# AUC 0.7485 means model correctly rankgs the locking loans higher
# about 74.85% of the time compared to non-locking loans

# Accuracy 0.6803
# 68.03% of all predictions were correct


## 9. Coefficient Review

For logistic regression, coefficients are in **log-odds** terms.

The important coefficient is `rate_diff`.

Expected sign:

```text
rate_diff coefficient should be negative
```

Reason:

```text
Higher rate_diff = worse rate compared with market = lower lock probability
```


In [9]:
feature_names = model.named_steps["preprocess"].get_feature_names_out()
coef = model.named_steps["model"].coef_[0]

coef_df = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient_log_odds": coef,
        "odds_multiplier": np.exp(coef)
    })
    .sort_values("coefficient_log_odds")
)

display(coef_df)


,feature,coefficient_log_odds,odds_multiplier
0,num__rate_diff,-0.9739,0.3776
3,num__loan_amount,-0.1721,0.8419
2,num__ltv,-0.1478,0.8626
1,num__fico,0.1582,1.1715
7,cat__occupancy_second_home,0.2327,1.2620
9,cat__market_volatility_medium,0.3518,1.4216
5,cat__purpose_refinance,0.4671,1.5954
6,cat__occupancy_primary,0.4686,1.5977
8,cat__market_volatility_low,0.5593,1.7495
4,cat__purpose_purchase,0.7053,2.0245


## 10. Translate Rate Elasticity Into Business Terms

A 0.125% margin/rate improvement means:

```text
rate_diff decreases by 0.125
```

Using the model's `rate_diff` coefficient, we can estimate the implied change in odds.


In [10]:
rate_feature_name = "num__rate_diff"

rate_coef = coef_df.loc[coef_df["feature"] == rate_feature_name, "coefficient_log_odds"].iloc[0]

rate_improvement = -0.125

log_odds_change = rate_coef * rate_improvement
odds_multiplier_0125 = np.exp(log_odds_change)

elasticity_summary = pd.DataFrame({
    "item": [
        "rate_diff coefficient",
        "rate improvement scenario",
        "change in log-odds",
        "implied odds multiplier"
    ],
    "value": [
        rate_coef,
        rate_improvement,
        log_odds_change,
        odds_multiplier_0125
    ]
})

display(elasticity_summary)


,item,value
0,rate_diff coefficient,-0.9739
1,rate improvement scenario,-0.1250
2,change in log-odds,0.1217
3,implied odds multiplier,1.1295


## 11. Scenario Test: Reduce Rate Margin by 0.125%

This estimates the expected conversion lift if we improve the borrower-facing rate by 0.125%.


In [11]:
scenario_df = X_model.copy()

baseline_prob = model.predict_proba(scenario_df)[:, 1]

scenario_df_0125 = scenario_df.copy()
scenario_df_0125["rate_diff"] = scenario_df_0125["rate_diff"] - 0.125

scenario_prob_0125 = model.predict_proba(scenario_df_0125)[:, 1]

scenario_results = pd.DataFrame({
    "scenario": ["baseline", "rate_margin_cut_0.125"],
    "avg_lock_probability": [
        baseline_prob.mean(),
        scenario_prob_0125.mean()
    ],
    "expected_locks": [
        baseline_prob.sum(),
        scenario_prob_0125.sum()
    ],
})

scenario_results["delta_expected_locks"] = (
    scenario_results["expected_locks"] - scenario_results.loc[0, "expected_locks"]
)

scenario_results["delta_lock_probability"] = (
    scenario_results["avg_lock_probability"] - scenario_results.loc[0, "avg_lock_probability"]
)

scenario_results["pct_lift_expected_locks"] = (
    scenario_results["expected_locks"] / scenario_results.loc[0, "expected_locks"] - 1
)

display(scenario_results)


,scenario,avg_lock_probability,expected_locks,delta_expected_locks,delta_lock_probability,pct_lift_expected_locks
0,baseline,0.5081,"152,437.7995",0.0000,0.0000,0.0000
1,rate_margin_cut_0.125,0.6621,"198,621.6801","46,183.8806",0.1539,0.3030


## 12. Margin / Revenue Tradeoff

Assumption:

```text
Current margin = 125 bps
New margin after 0.125 rate cut = 112.5 bps
```

This shows whether the expected lock lift is enough to offset the lower margin.


In [12]:
base_margin_bps = 125.0
cut_bps = 12.5
new_margin_bps = base_margin_bps - cut_bps

loan_amount_model = df.loc[X_model.index, "loan_amount"]

baseline_revenue = (loan_amount_model * (base_margin_bps / 10_000) * baseline_prob).sum()
scenario_revenue = (loan_amount_model * (new_margin_bps / 10_000) * scenario_prob_0125).sum()

revenue_summary = pd.DataFrame({
    "scenario": ["baseline", "rate_margin_cut_0.125"],
    "margin_bps": [base_margin_bps, new_margin_bps],
    "avg_lock_probability": [baseline_prob.mean(), scenario_prob_0125.mean()],
    "expected_locks": [baseline_prob.sum(), scenario_prob_0125.sum()],
    "expected_revenue": [baseline_revenue, scenario_revenue],
})

revenue_summary["delta_revenue"] = (
    revenue_summary["expected_revenue"] - revenue_summary.loc[0, "expected_revenue"]
)

revenue_summary["pct_revenue_change"] = (
    revenue_summary["expected_revenue"] / revenue_summary.loc[0, "expected_revenue"] - 1
)

display(revenue_summary)


,scenario,margin_bps,avg_lock_probability,expected_locks,expected_revenue,delta_revenue,pct_revenue_change
0,baseline,125.0000,0.5081,"152,437.7995","838,152,149.4317",0.0000,0.0000
1,rate_margin_cut_0.125,112.5000,0.6621,"198,621.6801","990,869,061.5085","152,716,912.0768",0.1822


## 13. Break-Even Lift

If margin drops from 125 bps to 112.5 bps, the required funded volume lift is:

```text
125 / 112.5 - 1 = 11.11%
```

So the model-estimated lift must beat that hurdle for the rate cut to pay for itself.


In [13]:
break_even_volume_lift = (base_margin_bps / new_margin_bps) - 1
model_expected_lift = (
    scenario_results.loc[1, "expected_locks"] / scenario_results.loc[0, "expected_locks"] - 1
)

break_even_summary = pd.DataFrame({
    "metric": [
        "break_even_required_volume_lift",
        "model_expected_lock_lift",
        "excess_lift_vs_break_even"
    ],
    "value": [
        break_even_volume_lift,
        model_expected_lift,
        model_expected_lift - break_even_volume_lift
    ]
})

display(break_even_summary)


,metric,value
0,break_even_required_volume_lift,0.1111
1,model_expected_lock_lift,0.3030
2,excess_lift_vs_break_even,0.1919


## 14. Segment-Level Elasticity

The rate cut may not be equally valuable across all loan types.

This section estimates which segments respond most to a 0.125 improvement.


In [14]:
segment_df = df.loc[X_model.index, ["purpose", "occupancy", "market_volatility", "loan_amount"]].copy()
segment_df["baseline_prob"] = baseline_prob
segment_df["scenario_prob_0125"] = scenario_prob_0125
segment_df["prob_lift"] = segment_df["scenario_prob_0125"] - segment_df["baseline_prob"]
segment_df["pct_lift"] = segment_df["scenario_prob_0125"] / segment_df["baseline_prob"] - 1

def summarize_segment(group_col):
    out = (
        segment_df
        .groupby(group_col)
        .agg(
            loans=("loan_amount", "count"),
            avg_loan_amount=("loan_amount", "mean"),
            baseline_lock_prob=("baseline_prob", "mean"),
            scenario_lock_prob=("scenario_prob_0125", "mean"),
            avg_prob_lift=("prob_lift", "mean"),
            avg_pct_lift=("pct_lift", "mean"),
            expected_locks_baseline=("baseline_prob", "sum"),
            expected_locks_scenario=("scenario_prob_0125", "sum"),
        )
        .reset_index()
    )
    out["expected_lock_lift"] = out["expected_locks_scenario"] - out["expected_locks_baseline"]
    out["expected_lock_lift_pct"] = out["expected_locks_scenario"] / out["expected_locks_baseline"] - 1
    return out.sort_values("expected_lock_lift_pct", ascending=False)

purpose_summary = summarize_segment("purpose")
occupancy_summary = summarize_segment("occupancy")
volatility_summary = summarize_segment("market_volatility")

display(purpose_summary)
display(occupancy_summary)
display(volatility_summary)


,purpose,loans,avg_loan_amount,baseline_lock_prob,scenario_lock_prob,avg_prob_lift,avg_pct_lift,expected_locks_baseline,expected_locks_scenario,expected_lock_lift,expected_lock_lift_pct
0,cash_out,44960,"450,895.9089",0.4017,0.5605,0.1588,0.5247,"18,060.2796","25,201.9863","7,141.7067",0.3954
2,refinance,90046,"451,747.8354",0.4961,0.6524,0.1563,0.4150,"44,674.5557","58,744.4285","14,069.8728",0.3149
1,purchase,164994,"452,091.1313",0.5437,0.6950,0.1514,0.3641,"89,702.9642","114,675.2653","24,972.3011",0.2784


,occupancy,loans,avg_loan_amount,baseline_lock_prob,scenario_lock_prob,avg_prob_lift,avg_pct_lift,expected_locks_baseline,expected_locks_scenario,expected_lock_lift,expected_lock_lift_pct
0,investment,44869,"451,928.1019",0.4332,0.5915,0.1583,0.4877,"19,435.4784","26,539.7267","7,104.2483",0.3655
2,second_home,30272,"452,187.6731",0.4815,0.6377,0.1562,0.4326,"14,575.6704","19,304.1461","4,728.4757",0.3244
1,primary,224859,"451,734.2091",0.5267,0.6794,0.1528,0.3827,"118,426.6506","152,777.8072","34,351.1566",0.2901


,market_volatility,loans,avg_loan_amount,baseline_lock_prob,scenario_lock_prob,avg_prob_lift,avg_pct_lift,expected_locks_baseline,expected_locks_scenario,expected_lock_lift,expected_lock_lift_pct
0,high,60124,"451,727.5447",0.4322,0.5908,0.1586,0.4886,"25,985.6019","35,520.2302","9,534.6283",0.3669
2,medium,104881,"452,052.3574",0.5040,0.6594,0.1553,0.4067,"52,863.0145","69,154.3253","16,291.3108",0.3082
1,low,134995,"451,656.1322",0.5451,0.6959,0.1508,0.3630,"73,589.1831","93,947.1247","20,357.9415",0.2766


## 15. Recommendation Logic

This is a simple decision rule for the business case.

The model is not the final decision by itself. It tells us whether the strategy is promising enough to test.


In [15]:
if model_expected_lift > break_even_volume_lift:
    recommendation = "Potentially attractive: estimated lock lift exceeds break-even volume lift."
elif model_expected_lift > 0:
    recommendation = "Positive conversion lift, but likely not enough to fully offset margin compression."
else:
    recommendation = "Not attractive: model does not estimate a positive conversion lift."

print(recommendation)


Potentially attractive: estimated lock lift exceeds break-even volume lift.


## 16. Export Results for Notebook 05

Notebook 05 can use these outputs to create the final business case / management summary.


In [16]:
scenario_results.to_csv("04_scenario_lock_lift_results.csv", index=False)
revenue_summary.to_csv("04_margin_revenue_summary.csv", index=False)
break_even_summary.to_csv("04_break_even_summary.csv", index=False)
purpose_summary.to_csv("04_segment_summary_by_purpose.csv", index=False)
occupancy_summary.to_csv("04_segment_summary_by_occupancy.csv", index=False)
volatility_summary.to_csv("04_segment_summary_by_market_volatility.csv", index=False)

print("Exported Notebook 04 outputs.")


Exported Notebook 04 outputs.
